In [1]:
import torch
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from seqeval.metrics import classification_report

In [2]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

device

'cuda'

In [3]:
import pickle

with open('./data/mimic_icd_data_dict.pkl', 'rb') as f:
    datadict = pickle.load(f)

In [4]:
class MimicIcdDataset(Dataset):
    def __init__(self, data_dict, tokenizer, tokenization_params, max_token_type_ids=100):
        """
        Setup parameters for the dataset
        Parameters
        ----------
        data_dict : dict with keys as patient ids
            training/validation data
        tokenizer : transformers tokenizer
            Tokenizer to do encoding of text
        tokenization_params : dict
            Parameters for tokenization passed to tokenizer. Note, do not set {'return_tensors': 'pt'} because this adds
            an extra dimension. Keep 'return_tensors' to default as this function does the torch tensor conversion when
            it returns the item.
            Suggested parameters: {'max_length': 512, 'truncation': True, 'padding': 'max_length'}
        """
        super().__init__()
        self.data = data_dict
        self.tokenizer = tokenizer
        self.params = tokenization_params
        self.patients = list(self.data.keys())
        self.len = len(self.data)
        self.max_token_type_ids = max_token_type_ids

    def __len__(self):
        return self.len

    def __getitem__(self, index):
        """Processing when returning an item from the Dataset. Converts text into encoding using tokenizer.
        Converts items from encoding to PyTorch tensors
        Parameters
        ----------
        index : int
            index of item to access
        Returns
        -------
        dict
            Output encoding from the tokenizer on the text.
            Relevant keys are ['input_ids', 'attention_mask', 'token_type_ids', 'label']
            What keys are present may depend on tokenizer parameters
        """
        k = self.patients[index]
        data = self.data[k]
        codes = data['icd_code']
        # Get coding embedding ids
        encoding = self.tokenizer(codes, **self.params)
        # Add token type ids for visit sequence
        num_codes_to_keep = min(self.params['max_length'] - 2, len(codes))
        encoding['token_type_ids'][1:num_codes_to_keep+1] = data['visit_order'][:num_codes_to_keep]
        encoding = {key: torch.as_tensor(val) for key, val in encoding.items()}
        return encoding


In [5]:
tokenization_params = {
    'max_length': 128,
    'truncation': True,
    'padding': 'max_length',
    'is_split_into_words': True,
    'return_special_tokens_mask': True
}

In [6]:
tokenizer = AutoTokenizer.from_pretrained('./data/tokenizer-icd-final')

In [7]:
dataset = MimicIcdDataset(datadict, tokenizer, tokenization_params)

In [8]:
BATCH_SIZE = 64

In [9]:
from transformers import DataCollatorForLanguageModeling

dataloader = DataLoader(dataset, shuffle=True, batch_size=BATCH_SIZE,
    collate_fn=DataCollatorForLanguageModeling(tokenizer),
    num_workers=4, pin_memory=True)

In [10]:
# model4 = torch.load('/home/vscode/workspace/trevor/bert_on_mimic/data/models/bert_for_icd_epoch_4.pt')

In [11]:
model3 = torch.load('/home/vscode/workspace/trevor/bert_on_mimic/data/models/bert_for_icd_epoch_3.pt')

In [12]:
model = model3
model.to(device)

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(26148, 768, padding_idx=0)
      (position_embeddings): Embedding(128, 768)
      (token_type_embeddings): Embedding(100, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True

In [13]:
model.eval()
loss_function = torch.nn.CrossEntropyLoss()

all_losses = []
all_preds = []
all_labels = []
all_acc = []

with torch.no_grad():
    for idx, batch in tqdm(enumerate(dataloader)):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)

        # B, N, V
        outputs = model(ids, mask, token_type_ids)
        
        # CrossEntropy expects (B, V, d1), so permute dims
        logits = outputs.logits.permute(0, 2, 1)
        loss = loss_function(logits, labels)
        all_losses.append(loss.item())

        # Get actual prediction
        preds = torch.argmax(logits, axis=1)

        # Get active labels
        active_mask = labels.view(-1) != -100
        active_labels = torch.masked_select(labels.view(-1), active_mask)
        active_preds = torch.masked_select(preds.view(-1), active_mask)
        acc = accuracy_score(active_preds.cpu(), active_labels.cpu())

        all_acc.append(acc)
        all_labels.extend([id.item() for id in active_labels])
        all_preds.extend([id.item() for id in active_preds])

2972it [22:13,  2.23it/s]


# Results for model after epoch 4

In [14]:
print(f"Average accuracy: {sum(all_acc) / len(all_acc):.2f}")
print(f"Average loss: {sum(all_losses) / len(all_losses):.2f}")

Average accuracy: 0.30
Average loss: 4.00


In [15]:
from sklearn.metrics import f1_score, precision_score, recall_score

In [16]:
args = (all_preds, all_labels)

for func, name in zip([f1_score, precision_score, recall_score], ['f1', 'precision', 'recall']):
    res = func(*args, average='weighted')
    print(f"{name}: {res:.2f}")

f1: 0.34


/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1327: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


precision: 0.46
recall: 0.30


/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1327: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


# Results for model after epoch 5

In [15]:
print(f"Average accuracy: {sum(all_acc) / len(all_acc):.2f}")
print(f"Average loss: {sum(all_losses) / len(all_losses):.2f}")

Average accuracy: 0.31
Average loss: 3.91


In [31]:
from sklearn.metrics import f1_score, precision_score, recall_score

In [34]:
args = (all_preds, all_labels)

for func, name in zip([f1_score, precision_score, recall_score], ['f1', 'precision', 'recall']):
    res = func(*args, average='weighted')
    print(f"{name}: {res:.2f}")

f1: 0.35


/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1327: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


precision: 0.46
recall: 0.31


/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1327: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
